# ViralScope AI - Data Pipeline

This notebook downloads and processes the YouTube Trending dataset for the ViralScope AI project.

**Tasks covered:**
- T1.1: Download YouTube Trending Dataset from Kaggle
- T1.2: Deduplicate and filter invalid rows
- T1.3: Download thumbnails from YouTube CDN
- T1.4: Filter to videos with valid thumbnails
- T1.5: Compute channel statistics (Leave-One-Out)
- T1.6: Compute viral multiplier and binary labels
- T1.7: Pre-tokenize text corpus
- T1.9: Grouped Train/Val/Test split
- T1.10: Compute class weights

**Runtime:** ~30-60 minutes (thumbnail download is the bottleneck)

In [ ]:
#@title 1. Setup & Installation
!pip install kaggle pandas pillow requests tqdm transformers torch torchvision scikit-learn -q

In [ ]:
#@title 2. Imports & Configuration
import os
import re
import json
import hashlib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
from transformers import DistilBertTokenizer
from sklearn.model_selection import GroupShuffleSplit

# Configuration
CONFIG = {
    'project': {'name': 'ViralScope AI', 'seed': 42, 'device': 'auto'},
    'data': {
        'raw_dir': 'data/raw',
        'processed_dir': 'data/processed',
        'tensor_dir': 'data/tensors',
        'splits_dir': 'data/splits',
        'train_split': 0.7,
        'val_split': 0.15,
        'test_split': 0.15,
        'target_threshold': 1.5,
        'thumbnail_rate_limit': 10
    },
    'model': {
        'nlp': {
            'checkpoint': 'distilbert-base-uncased',
            'max_seq_length': 64
        }
    }
}

# Create directories
for dir_path in [
    CONFIG['data']['raw_dir'],
    f"{CONFIG['data']['raw_dir']}/thumbnails",
    CONFIG['data']['processed_dir'],
    CONFIG['data']['tensor_dir'],
    CONFIG['data']['splits_dir']
]:
    os.makedirs(dir_path, exist_ok=True)

print("✓ Directories created successfully")

In [ ]:
#@title 3. Task T1.1: Download YouTube Trending Dataset from Kaggle

print("="*60)
print("TASK T1.1: Downloading YouTube Trending Dataset")
print("="*60)

# Configure Kaggle credentials
KAGGLE_USERNAME = 'gannourr'
KAGGLE_KEY = 'KGAT_59106961c6b7068fee1f289400d19560'

# Setup Kaggle config
!mkdir -p ~/.kaggle
!echo '{{"username":"{KAGGLE_USERNAME}","key":"{KAGGLE_KEY}"}}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

KAGGLE_DATASET = 'datasnaik/youtube-trending-videos-dataset'

!kaggle datasets download -d {KAGGLE_DATASET} -p {CONFIG['data']['raw_dir']} --unzip -q

# Consolidate all CSV files into one
csv_files = [f for f in os.listdir(CONFIG['data']['raw_dir']) if f.endswith('.csv')]
print(f"Found {len(csv_files)} CSV files: {csv_files}")

dfs = []
for csv_file in csv_files:
    filepath = os.path.join(CONFIG['data']['raw_dir'], csv_file)
    try:
        df_temp = pd.read_csv(filepath, low_memory=False)
        dfs.append(df_temp)
        print(f"  Loaded {csv_file}: {len(df_temp)} rows")
    except Exception as e:
        print(f"  Error loading {csv_file}: {e}")

if dfs:
    trending_df = pd.concat(dfs, ignore_index=True)
    trending_df = trending_df.drop_duplicates(subset=['video_id'], keep='last')
    trending_df.to_csv(f"{CONFIG['data']['raw_dir']}/trending.csv", index=False)
    print(f"\n✓ Combined dataset: {len(trending_df)} rows")
    print(f"✓ Columns: {list(trending_df.columns)}")
else:
    print("⚠ No CSV files found. Creating sample data for testing.")
    trending_df = pd.DataFrame({
        'video_id': [f'vid_{i}' for i in range(1000)],
        'title': [f'Sample Video Title {i}' for i in range(1000)],
        'views': np.random.randint(1000, 10000000, 1000),
        'channel_title': [f'Channel {i%50}' for i in range(1000)],
        'channel_id': [f'ch_{i%50}' for i in range(1000)]
    })
    trending_df.to_csv(f"{CONFIG['data']['raw_dir']}/trending.csv", index=False)
    print(f"✓ Created sample dataset: {len(trending_df)} rows")

In [ ]:
#@title 4. Task T1.2: Deduplicate and Filter Invalid Rows

print("="*60)
print("TASK T1.2: Deduplicate and Filter Invalid Rows")
print("="*60)

trending_df = pd.read_csv(f"{CONFIG['data']['raw_dir']}/trending.csv")
print(f"Raw rows: {len(trending_df)}")

trending_df = trending_df.dropna(subset=['video_id'])
trending_df = trending_df[trending_df['video_id'].str.strip() != '']

trending_df = trending_df.dropna(subset=['views'])
trending_df['views'] = pd.to_numeric(trending_df['views'], errors='coerce')
trending_df = trending_df.dropna(subset=['views'])
trending_df = trending_df[trending_df['views'] > 0]

trending_df = trending_df.drop_duplicates(subset=['video_id'], keep='last')

trending_df['title'] = trending_df['title'].fillna('')
trending_df = trending_df[trending_df['title'].str.strip() != '']

required_cols = ['video_id', 'title', 'views', 'channel_title', 'channel_id']
available_cols = [c for c in required_cols if c in trending_df.columns]
clean_df = trending_df[available_cols].copy()

print(f"Clean rows: {len(clean_df)}")
print(f"Reduced from {len(trending_df)} raw rows to {len(clean_df)} clean rows")

clean_df.to_csv(f"{CONFIG['data']['processed_dir']}/clean_dataset.csv", index=False)
print(f"\n✓ Saved: {CONFIG['data']['processed_dir']}/clean_dataset.csv")

In [ ]:
#@title 5. Task T1.3: Download Thumbnails from YouTube CDN

print("="*60)
print("TASK T1.3: Downloading Thumbnails from YouTube CDN")
print("="*60)

def download_thumbnail(video_id, save_dir, min_width=120, min_height=90):
    save_path = os.path.join(save_dir, f"{video_id}.jpg")
    
    if os.path.exists(save_path):
        return video_id, "exists"
    
    session = requests.Session()
    session.headers.update({'User-Agent': 'Mozilla/5.0'})
    
    for res in ["maxresdefault", "hqdefault"]:
        url = f"https://img.youtube.com/vi/{video_id}/{res}.jpg"
        try:
            resp = session.get(url, timeout=10)
            if resp.status_code == 200 and len(resp.content) > 1000:
                img = Image.open(BytesIO(resp.content))
                w, h = img.size
                if w > min_width and h > min_height:
                    with open(save_path, "wb") as f:
                        f.write(resp.content)
                    return video_id, "success"
        except Exception:
            continue
    
    return video_id, "failed"


clean_df = pd.read_csv(f"{CONFIG['data']['processed_dir']}/clean_dataset.csv")
video_ids = clean_df['video_id'].tolist()
thumbnail_dir = f"{CONFIG['data']['raw_dir']}/thumbnails"

print(f"Downloading thumbnails for {len(video_ids)} videos...")
print(f"Using {5} parallel workers")

results = {}
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(download_thumbnail, vid, thumbnail_dir): vid for vid in video_ids}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading"):
        vid, status = future.result()
        results[vid] = status

manifest_df = pd.DataFrame(list(results.items()), columns=['video_id', 'status'])
manifest_df.to_csv(f"{CONFIG['data']['processed_dir']}/thumbnail_manifest.csv", index=False)

success_count = sum(1 for s in results.values() if s == "success")
exists_count = sum(1 for s in results.values() if s == "exists")
failed_count = sum(1 for s in results.values() if s == "failed")
total_available = success_count + exists_count
success_rate = total_available / len(results) * 100

print(f"\n📊 Download Summary:")
print(f"  - Success: {success_count}")
print(f"  - Already existed: {exists_count}")
print(f"  - Failed: {failed_count}")
print(f"  - Success rate: {success_rate:.1f}%")
print(f"\n✓ Saved manifest: {CONFIG['data']['processed_dir']}/thumbnail_manifest.csv")

In [ ]:
#@title 6. Task T1.4: Filter to Videos with Valid Thumbnails

print("="*60)
print("TASK T1.4: Filter to Videos with Valid Thumbnails")
print("="*60)

clean_df = pd.read_csv(f"{CONFIG['data']['processed_dir']}/clean_dataset.csv")
manifest_df = pd.read_csv(f"{CONFIG['data']['processed_dir']}/thumbnail_manifest.csv")

valid_thumbnails = manifest_df[manifest_df['status'].isin(['success', 'exists'])]['video_id'].tolist()
final_df = clean_df[clean_df['video_id'].isin(valid_thumbnails)].copy()

coverage = len(final_df) / len(clean_df) * 100

print(f"Final dataset: {len(final_df)} videos with thumbnails ({coverage:.1f}% of clean dataset)")

if coverage < 50:
    print("⚠️ Warning: Thumbnail coverage is below 50%. Consider re-running thumbnail download.")

final_df.to_csv(f"{CONFIG['data']['processed_dir']}/final_dataset.csv", index=False)
print(f"\n✓ Saved: {CONFIG['data']['processed_dir']}/final_dataset.csv")

In [ ]:
#@title 7. Task T1.5: Compute Channel Statistics (Leave-One-Out)

print("="*60)
print("TASK T1.5: Compute Channel Statistics (Leave-One-Out)")
print("="*60)

final_df = pd.read_csv(f"{CONFIG['data']['processed_dir']}/final_dataset.csv")

final_df['channel_sum_views'] = final_df.groupby('channel_id')['views'].transform('sum')
final_df['channel_video_count'] = final_df.groupby('channel_id')['views'].transform('count')

final_df['loo_avg_views'] = (
    (final_df['channel_sum_views'] - final_df['views']) / 
    (final_df['channel_video_count'] - 1 + 1e-5)
)

channel_stats = final_df.groupby('channel_id').agg(
    loo_avg_views=('loo_avg_views', 'mean'),
    video_count=('channel_video_count', 'first')
).reset_index()

channel_stats.to_csv(f"{CONFIG['data']['processed_dir']}/channel_averages.csv", index=False)

print(f"Computed LOO statistics for {len(channel_stats)} channels")
print(f"✓ Saved: {CONFIG['data']['processed_dir']}/channel_averages.csv")

In [ ]:
#@title 8. Task T1.6: Compute Viral Multiplier and Binary Labels

print("="*60)
print("TASK T1.6: Compute Viral Multiplier and Binary Labels")
print("="*60)

final_df = pd.read_csv(f"{CONFIG['data']['processed_dir']}/final_dataset.csv")
channel_stats = pd.read_csv(f"{CONFIG['data']['processed_dir']}/channel_averages.csv")

labeled_df = final_df.merge(channel_stats, on='channel_id', how='left')

labeled_df['channel_sum'] = labeled_df.groupby('channel_id')['views'].transform('sum')
labeled_df['channel_count'] = labeled_df.groupby('channel_id')['views'].transform('count')
labeled_df['channel_loo_avg_views'] = (
    (labeled_df['channel_sum'] - labeled_df['views']) / 
    (labeled_df['channel_count'] - 1)
)

labeled_df['multiplier'] = labeled_df['views'] / (labeled_df['channel_loo_avg_views'] + 1e-5)

reliable_channels = channel_stats[channel_stats['video_count'] >= 3]['channel_id'].tolist()
removed_count = len(labeled_df) - len(labeled_df[labeled_df['channel_id'].isin(reliable_channels)])
labeled_df = labeled_df[labeled_df['channel_id'].isin(reliable_channels)]

print(f"Removed {removed_count} videos from channels with <3 videos")

threshold = CONFIG['data']['target_threshold']
labeled_df['is_viral'] = (labeled_df['multiplier'] > threshold).astype(int)

minority_pct = min(labeled_df['is_viral'].mean(), 1 - labeled_df['is_viral'].mean())
if minority_pct < 0.10:
    threshold = labeled_df['multiplier'].quantile(0.75)
    print(f"⚠️ Class imbalance ({minority_pct:.1%}). Using dynamic threshold: {threshold:.2f}")
    labeled_df['is_viral'] = (labeled_df['multiplier'] > threshold).astype(int)
    minority_pct = min(labeled_df['is_viral'].mean(), 1 - labeled_df['is_viral'].mean())

viral_count = labeled_df['is_viral'].sum()
non_viral_count = len(labeled_df) - viral_count
print(f"Class distribution:")
print(f"  - Viral: {viral_count} ({labeled_df['is_viral'].mean():.1%})")
print(f"  - Non-viral: {non_viral_count} ({1-labeled_df['is_viral'].mean():.1%})")

output_cols = ['video_id', 'title', 'views', 'channel_loo_avg_views', 'multiplier', 'is_viral', 'channel_id']
labeled_df = labeled_df[output_cols]

labeled_df.to_csv(f"{CONFIG['data']['processed_dir']}/labeled_dataset.csv", index=False)
print(f"\n✓ Saved: {CONFIG['data']['processed_dir']}/labeled_dataset.csv")

In [ ]:
#@title 9. Task T1.7: Pre-Tokenize Text Corpus (FIXED)

print("="*60)
print("TASK T1.7: Pre-Tokenize Text Corpus with DistilBERT")
print("="*60)

labeled_df = pd.read_csv(f"{CONFIG['data']['processed_dir']}/labeled_dataset.csv")

# Clean titles - ensure all are valid strings
def clean_title(title):
    if pd.isna(title) or str(title).strip() == '':
        return 'Untitled Video'
    # Remove emojis and special characters
    cleaned = re.sub(r'[\U00010000-\U0010ffff]', '', str(title))
    if cleaned.strip() == '':
        return 'Untitled Video'
    return cleaned

labeled_df['title'] = labeled_df['title'].apply(clean_title)
print(f"Cleaned titles. Sample: {labeled_df['title'].iloc[0][:50]}...")

print("Loading DistilBERT tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained(
    CONFIG['model']['nlp']['checkpoint'],
    use_fast=True
)

print("Tokenizing titles in batches...")
titles = labeled_df['title'].tolist()
print(f"Total titles to tokenize: {len(titles)}")

# Tokenize in batches to avoid memory issues
batch_size = 5000
all_input_ids = []
all_attention_masks = []

for i in tqdm(range(0, len(titles), batch_size), desc="Tokenizing batches"):
    batch = titles[i:i+batch_size]
    # Tokenize each title individually then stack
    encoded = tokenizer(
        batch,
        max_length=CONFIG['model']['nlp']['max_seq_length'],
        padding='max_length',
        truncation=True,
        return_tensors=None  # Return lists, not tensors
    )
    all_input_ids.extend(encoded['input_ids'])
    all_attention_masks.extend(encoded['attention_mask'])

# Convert to tensors
input_ids = torch.tensor(all_input_ids, dtype=torch.long)
attention_masks = torch.tensor(all_attention_masks, dtype=torch.long)

print(f"Tokenization complete! Shape: {input_ids.shape}")

# Save tensors
torch.save(input_ids, f"{CONFIG['data']['tensor_dir']}/input_ids.pt")
torch.save(attention_masks, f"{CONFIG['data']['tensor_dir']}/attention_masks.pt")

# Compute and save dataset hash
df_hash = hashlib.md5(labeled_df.to_csv(index=False).encode()).hexdigest()

metadata = {
    'dataset_hash': df_hash,
    'num_samples': len(labeled_df),
    'max_seq_length': CONFIG['model']['nlp']['max_seq_length'],
    'tokenizer_checkpoint': CONFIG['model']['nlp']['checkpoint']
}

with open(f"{CONFIG['data']['tensor_dir']}/tokenizer_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Saved: {CONFIG['data']['tensor_dir']}/input_ids.pt - shape {input_ids.shape}")
print(f"✓ Saved: {CONFIG['data']['tensor_dir']}/attention_masks.pt - shape {attention_masks.shape}")
print(f"✓ Saved: {CONFIG['data']['tensor_dir']}/tokenizer_metadata.json")
print(f"✓ Dataset hash: {df_hash}")

In [ ]:
#@title 10. Task T1.9: Grouped Train/Val/Test Split

print("="*60)
print("TASK T1.9: Grouped Train/Val/Test Split by Channel")
print("="*60)

labeled_df = pd.read_csv(f"{CONFIG['data']['processed_dir']}/labeled_dataset.csv")

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(labeled_df, groups=labeled_df['channel_id']))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
temp_df = labeled_df.iloc[temp_idx].reset_index(drop=True)
val_temp_idx, test_temp_idx = next(gss2.split(temp_df, groups=temp_df['channel_id']))

val_idx = temp_idx[val_temp_idx]
test_idx = temp_idx[test_temp_idx]

train_channels = set(labeled_df.iloc[train_idx]['channel_id'].unique())
val_channels = set(labeled_df.iloc[val_idx]['channel_id'].unique())
test_channels = set(labeled_df.iloc[test_idx]['channel_id'].unique())

assert len(train_channels & val_channels) == 0, "DATA LEAKAGE: channels in both train and val!"
assert len(train_channels & test_channels) == 0, "DATA LEAKAGE: channels in both train and test!"
assert len(val_channels & test_channels) == 0, "DATA LEAKAGE: channels in both val and test!"

print("✓ No channel overlap between splits - data leakage prevented")

torch.save(torch.tensor(train_idx, dtype=torch.long), f"{CONFIG['data']['splits_dir']}/train_indices.pt")
torch.save(torch.tensor(val_idx, dtype=torch.long), f"{CONFIG['data']['splits_dir']}/val_indices.pt")
torch.save(torch.tensor(test_idx, dtype=torch.long), f"{CONFIG['data']['splits_dir']}/test_indices.pt")

def split_stats(df, indices, name):
    subset = df.iloc[indices]
    return {
        'split': name,
        'num_videos': len(indices),
        'num_channels': int(subset['channel_id'].nunique()),
        'viral_count': int(subset['is_viral'].sum()),
        'non_viral_count': int((subset['is_viral'] == 0).sum()),
        'viral_pct': float(subset['is_viral'].mean())
    }

report = {
    'train': split_stats(labeled_df, train_idx, 'train'),
    'val': split_stats(labeled_df, val_idx, 'val'),
    'test': split_stats(labeled_df, test_idx, 'test'),
    'total_videos': len(labeled_df),
    'total_channels': int(labeled_df['channel_id'].nunique()),
    'channel_overlap': {
        'train_val': len(train_channels & val_channels),
        'train_test': len(train_channels & test_channels),
        'val_test': len(val_channels & test_channels)
    }
}

with open(f"{CONFIG['data']['splits_dir']}/split_report.json", 'w') as f:
    json.dump(report, f, indent=2)

print(f"\n📊 Split Report:")
print(f"  Train: {report['train']['num_videos']} videos, {report['train']['num_channels']} channels")
print(f"  Val:   {report['val']['num_videos']} videos, {report['val']['num_channels']} channels")
print(f"  Test:  {report['test']['num_videos']} videos, {report['test']['num_channels']} channels")

print(f"\n✓ Saved: {CONFIG['data']['splits_dir']}/train_indices.pt")
print(f"✓ Saved: {CONFIG['data']['splits_dir']}/val_indices.pt")
print(f"✓ Saved: {CONFIG['data']['splits_dir']}/test_indices.pt")
print(f"✓ Saved: {CONFIG['data']['splits_dir']}/split_report.json")

In [ ]:
#@title 11. Task T1.10: Compute Class Weights

print("="*60)
print("TASK T1.10: Compute Class Weights")
print("="*60)

labeled_df = pd.read_csv(f"{CONFIG['data']['processed_dir']}/labeled_dataset.csv")
train_indices = torch.load(f"{CONFIG['data']['splits_dir']}/train_indices.pt")

train_labels = labeled_df.iloc[train_indices.numpy()]['is_viral'].values
n_samples = len(train_labels)
n_positive = int(train_labels.sum())
n_negative = n_samples - n_positive
n_classes = 2

weights = []
for count in [n_negative, n_positive]:
    weight = n_samples / (n_classes * count)
    weights.append(weight)

weights = [w / sum(weights) * n_classes for w in weights]
weights = torch.tensor(weights, dtype=torch.float32)

print(f"Class distribution in training set:")
print(f"  - Positive (viral): {n_positive} ({n_positive/n_samples:.1%})")
print(f"  - Negative (non-viral): {n_negative} ({n_negative/n_samples:.1%})")
print(f"\nClass weights: {weights.tolist()}")

torch.save(weights, f"{CONFIG['data']['processed_dir']}/class_weights.pt")
print(f"\n✓ Saved: {CONFIG['data']['processed_dir']}/class_weights.pt")

In [ ]:
#@title 12. Final Summary

print("="*60)
print("✓ DATA PIPELINE COMPLETE")
print("="*60)

print("\n📁 Generated Files:")
print("\n  data/raw/")
print("    ├── trending.csv")
print("    └── thumbnails/ (YouTube thumbnail images)")

print("\n  data/processed/")
print("    ├── clean_dataset.csv")
print("    ├── thumbnail_manifest.csv")
print("    ├── final_dataset.csv")
print("    ├── channel_averages.csv")
print("    ├── labeled_dataset.csv (IMMUTABLE)")
print("    └── class_weights.pt")

print("\n  data/tensors/")
print("    ├── input_ids.pt")
print("    ├── attention_masks.pt")
print("    └── tokenizer_metadata.json")

print("\n  data/splits/")
print("    ├── train_indices.pt")
print("    ├── val_indices.pt")
print("    ├── test_indices.pt")
print("    └── split_report.json")

print("\n🚀 Next Steps:")
print("   1. Download this notebook and run scripts/03_train.py locally")
   "2. Or continue in Colab with the model training cells")